# Framework-style SMD loading

This template uses the public notebook-facing API added in `src/data` and `src/adapters`.
It keeps the current registry-driven script path intact while giving notebooks a short import path.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

In [ ]:
from src.adapters import MomentWindowAdapter
from src.data import (
    flatten_windows_for_baseline,
    load_smd_data,
    point_labels_to_window_labels,
)

data = load_smd_data(
    root="data",
    window_size=100,
    stride=10,
    batch_size=32,
    download=False,
)
data

In [ ]:
train_batch = next(iter(data.loaders["train"]))
print(train_batch["x"].shape)
print(train_batch["point_labels"].shape if train_batch["point_labels"] is not None else None)
print(train_batch["meta"][0])

In [ ]:
baseline_x = flatten_windows_for_baseline(train_batch["x"])
window_labels = None
if train_batch["point_labels"] is not None:
    window_labels = point_labels_to_window_labels(train_batch["point_labels"])

print(baseline_x.shape)
print(window_labels.shape if window_labels is not None else None)

In [ ]:
moment_adapter = MomentWindowAdapter(context_length=512)
prepared = moment_adapter.prepare_batch(train_batch)
print(prepared["x_enc"].shape)
print(prepared["input_mask"].shape)
print(prepared["observed_window_length"])